# L11 · Domain Randomization

This lab connects the complete concept from the lecture to one deliberately small practice path:

```text
predict a bounded domain → render fixed Layer-A seeds → inspect Layer-B guardrails
→ record four successful DR episodes → audit data, attempts, provenance, and split risk
```

The current implementation covers flat colors, FOV, shared friction, per-object mass, and the static world-camera pose. It does not implement texture, lighting, geometry, actuator, timing, sensor-corruption, disturbance, or adaptive-distribution randomization.

## Before you run

Install the complete course dependencies and finish L09 first. By default this notebook reads `ROBO_GENESIS_DATASETS_DIR/l09_banana_demo`, writes `l11_banana_dr`, and places preview frames under `ROBO_GENESIS_OUTPUTS_DIR/eval_results/l11_dr_preview`. Use `RG101_L11_BASELINE_ROOT`, `RG101_L11_DATASET_ROOT`, and `RG101_L11_PREVIEW_ROOT` only for explicit compatible locations.

`ROBO_GENESIS_BACKEND=auto` selects the verified AMD path when ROCm is available and otherwise uses CPU. `ROBO_GENESIS_RENDER` defaults to `1`. Setting it to `0` runs only configuration, schedule, guardrail, and command checks; it does not initialize Genesis, create a dataset, or complete the lab.

Existing L11 output roots stop the normal path. Set `RG101_L11_OVERWRITE=1` before kernel startup only when you intend to replace the exact validated `l11_banana_dr` and `l11_dr_preview` roots. Generated data, images, videos, caches, and provenance stay outside Git.

In [ ]:
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.domain_provenance import (
    DOMAIN_PROVENANCE_RELATIVE_PATH,
    committed_episode_records,
    load_domain_provenance,
    plan_domain_split,
    plan_seed_schedule,
    validate_dr_collection_config,
)
from robo_genesis.paths import DATASETS_DIR, OUTPUTS_DIR

lesson = load_course_manifest().lesson('L11')
assert lesson.slug == 'domain-randomization'
assert lesson.duration_minutes == 90
assert lesson.hardware.value == 'gpu-recommended'
assert lesson.status.value == 'planned'

backend_mode = os.environ.get('ROBO_GENESIS_BACKEND', 'auto').strip().lower()
if backend_mode not in {'auto', 'cpu'}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get('ROBO_GENESIS_RENDER', '1').strip()
if render_value not in {'0', '1'}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == '1'
overwrite_enabled = os.environ.get('RG101_L11_OVERWRITE', '0').strip() == '1'

BASELINE_REPO_ID = os.environ.get(
    'RG101_L11_BASELINE_REPO_ID', 'local/l09_banana_demo'
).strip()
if not BASELINE_REPO_ID:
    raise ValueError('RG101_L11_BASELINE_REPO_ID must not be empty')
DR_REPO_ID = 'local/l11_banana_dr'
BASE_SEED = 1100
APPEARANCE_BASE_SEED = 0
DATASET_FPS = 5
IMG_WH = (640, 360)
TARGET_EPISODES = 4
DR_REBUILD_EVERY = 2
MAX_ATTEMPTS = 10

baseline_value = os.environ.get('RG101_L11_BASELINE_ROOT')
baseline_root = (
    Path(baseline_value).expanduser().resolve()
    if baseline_value
    else (DATASETS_DIR / 'l09_banana_demo').resolve()
)
dataset_value = os.environ.get('RG101_L11_DATASET_ROOT')
dataset_root = (
    Path(dataset_value).expanduser().resolve()
    if dataset_value
    else (DATASETS_DIR / 'l11_banana_dr').resolve()
)
preview_value = os.environ.get('RG101_L11_PREVIEW_ROOT')
preview_root = (
    Path(preview_value).expanduser().resolve()
    if preview_value
    else (OUTPUTS_DIR / 'eval_results' / 'l11_dr_preview').resolve()
)
if dataset_root.name != 'l11_banana_dr':
    raise ValueError(f'Unsafe L11 dataset path: {dataset_root}')
if preview_root.name != 'l11_dr_preview':
    raise ValueError(f'Unsafe L11 preview path: {preview_root}')

cache_root = (OUTPUTS_DIR / 'l11_cache').resolve()
for variable, relative in {
    'HF_DATASETS_CACHE': 'huggingface_datasets',
    'XDG_CACHE_HOME': 'xdg',
    'MPLCONFIGDIR': 'matplotlib',
}.items():
    os.environ.setdefault(variable, str(cache_root / relative))
    Path(os.environ[variable]).mkdir(parents=True, exist_ok=True)

if render_enabled:
    missing_baseline = [
        path
        for path in (
            baseline_root / 'meta' / 'info.json',
            baseline_root / 'meta' / 'stats.json',
        )
        if not path.is_file()
    ]
    if missing_baseline:
        raise FileNotFoundError(
            f'L11 baseline is incomplete at {baseline_root}: {missing_baseline}. '
            'Run the complete L09 notebook or set RG101_L11_BASELINE_ROOT. '
            'No download or fallback dataset is attempted.'
        )
    for output_root in (dataset_root, preview_root):
        if output_root.exists() and not overwrite_enabled:
            raise FileExistsError(
                f'{output_root} already exists; choose fresh roots or set '
                'RG101_L11_OVERWRITE=1 before restarting the kernel'
            )
    if preview_root.exists():
        shutil.rmtree(preview_root)

    required_versions = {
        'genesis-world': '1.3.3',
        'lerobot': '0.6.0',
        'av': '15.1.0',
        'pyarrow': '25.0.0',
    }
    installed_versions = {
        name: importlib.metadata.version(name) for name in required_versions
    }
    version_mismatches = {
        name: (installed_versions[name], expected)
        for name, expected in required_versions.items()
        if installed_versions[name] != expected
    }
    if version_mismatches:
        raise RuntimeError(f'Dependency version mismatch: {version_mismatches}')

    import torch

    use_cpu_backend = backend_mode == 'cpu' or not (
        bool(torch.version.hip) and bool(torch.cuda.is_available())
    )
    resolved_backend = 'cpu' if use_cpu_backend else 'amdgpu'
else:
    installed_versions = {}
    use_cpu_backend = backend_mode == 'cpu'
    resolved_backend = 'not initialized'

print(f'L11: {lesson.duration_minutes} min, hardware={lesson.hardware.value}, status={lesson.status.value}')
print(f'Requested backend={backend_mode}; resolved path={resolved_backend}; render={render_enabled}')
print(f'Baseline root: {baseline_root}')
print(f'L11 dataset root: {dataset_root}')
print(f'Preview root: {preview_root}')
if not render_enabled:
    print('DIAGNOSTIC MODE: no Genesis, camera, preview, writer, dataset, or readback')

## 1. Predict the domain and transaction

Before rendering, identify what changes, when it is sampled, whether it can affect expert success, and what must be recorded. The example schedule deliberately begins with a failed attempt: runtime seeds follow attempts, while the appearance-domain counter follows committed successes.

In [ ]:
from robo_genesis.scene_config import WORLD_CAM_FOV, WRIST_CAM_FOV

collection_config = validate_dr_collection_config(
    episodes=TARGET_EPISODES,
    max_attempts=MAX_ATTEMPTS,
    fps=DATASET_FPS,
    image_width=IMG_WH[0],
    image_height=IMG_WH[1],
    dr_rebuild_every=DR_REBUILD_EVERY,
    table_color_jitter=0.15,
    fov_jitter_deg=2.0,
    camera_fovs_deg=(WORLD_CAM_FOV, WRIST_CAM_FOV),
    friction_ratio_range=(0.7, 1.3),
    mass_ratio_range=(0.8, 1.2),
    cam_pos_jitter=0.01,
    cam_lookat_jitter=0.02,
)
example_schedule = plan_seed_schedule(
    [False, True, True, True, True],
    runtime_base_seed=BASE_SEED,
    appearance_base_seed=APPEARANCE_BASE_SEED,
    rebuild_every=collection_config['dr_rebuild_every'],
)
randomization_families = {
    'Layer A / build': ('table color', 'flat object color', 'world+wrist FOV'),
    'Layer B / episode reset': (
        'shared contact friction ratio',
        'per-object mass ratio',
        'static world-camera pose',
    ),
    'concept only in this lab': (
        'texture/material/light',
        'geometry/kinematics',
        'actuator/timing',
        'sensor corruption',
        'external disturbance',
        'adaptive distribution',
    ),
}
contract_checks = {
    'bounded_configuration': collection_config['friction_ratio_range'] == [0.7, 1.3]
    and collection_config['mass_ratio_range'] == [0.8, 1.2],
    'failed_attempt_consumes_runtime_seed': [
        row['runtime_seed'] for row in example_schedule
    ] == [1100, 1101, 1102, 1103, 1104],
    'appearance_advances_after_two_successes': [
        row['appearance_domain_index'] for row in example_schedule
    ] == [0, 0, 0, 1, 1],
    'failed_attempt_not_committed': example_schedule[0]['committed_episode_index'] is None,
}
failed_contract = [name for name, passed in contract_checks.items() if not passed]
if failed_contract:
    raise AssertionError('Randomization contract checks failed: ' + ', '.join(failed_contract))

print('Current implemented subset:')
for lifetime, parameters in randomization_families.items():
    print(f'  {lifetime}: {", ".join(parameters)}')
print('Predicted failure/success schedule:')
print(json.dumps(example_schedule, indent=2))

## 2. Render fixed Layer-A domains

The installed preview tool starts one isolated process for the DR-off baseline and each requested appearance seed. Keep `PREVIEW_PROFILE='combined'` for the main lab. For the final exercise, change only that line to `table_color`, `object_color`, or `fov`, then rerun Sections 2 and 3.

In [ ]:
PREVIEW_PROFILE = 'combined'  # Exercise: table_color, object_color, or fov.
preview_profiles = {
    'combined': ['--object-color', '--table-jitter', '0.15', '--fov-jitter', '2.0'],
    'table_color': ['--table-jitter', '0.15', '--fov-jitter', '0.0'],
    'object_color': ['--object-color', '--table-jitter', '0.0', '--fov-jitter', '0.0'],
    'fov': ['--table-jitter', '0.0', '--fov-jitter', '2.0'],
}
if PREVIEW_PROFILE not in preview_profiles:
    raise ValueError(f'PREVIEW_PROFILE must be one of {tuple(preview_profiles)}')
preview_run_root = (
    preview_root
    if PREVIEW_PROFILE == 'combined'
    else preview_root / f'exercise_{PREVIEW_PROFILE}'
)

preview_command = [
    sys.executable,
    '-m',
    'robo_genesis.tools.dr_preview',
    '--seeds',
    '0',
    '1',
    '--output-dir',
    str(preview_run_root),
    *preview_profiles[PREVIEW_PROFILE],
]
if use_cpu_backend:
    preview_command.append('--cpu')

preview_frame_paths = {
    'baseline': preview_run_root / 'frames' / 'world_baseline.png',
    'seed 0': preview_run_root / 'frames' / 'world_seed0.png',
    'seed 1': preview_run_root / 'frames' / 'world_seed1.png',
}
preview_checks = {
    'command_uses_installed_module': preview_command[:3]
    == [sys.executable, '-m', 'robo_genesis.tools.dr_preview'],
    'fixed_appearance_seeds': preview_command[
        preview_command.index('--seeds') + 1 : preview_command.index('--seeds') + 3
    ] == ['0', '1'],
    'recognized_preview_profile': PREVIEW_PROFILE in preview_profiles,
}

print('Preview command:', ' '.join(preview_command))
if render_enabled:
    subprocess.run(preview_command, check=True, env=os.environ.copy())
    preview_checks = {
        **preview_checks,
        'all_world_frames_written': all(path.is_file() for path in preview_frame_paths.values()),
        'world_montage_written': (preview_run_root / 'world_montage.png').is_file(),
    }
    failed_preview = [name for name, passed in preview_checks.items() if not passed]
    if failed_preview:
        raise AssertionError('Appearance preview checks failed: ' + ', '.join(failed_preview))
else:
    preview_checks['diagnostic_mode_requested'] = True
    print('SKIP — preview subprocess was not started')

## 3. Inspect the rendered observations

Read the actual baseline and seed frames. Pixel differences show that the selected knobs changed an observation; they do not establish realism, sufficient coverage, policy benefit, or sim-to-real transfer.

In [ ]:
appearance_checks = {'diagnostic_mode_requested': not render_enabled}
appearance_summary = {}

if render_enabled:
    import imageio.v2 as imageio
    import matplotlib.pyplot as plt

    preview_images = {
        label: np.asarray(imageio.imread(path))
        for label, path in preview_frame_paths.items()
    }
    reference_shape = preview_images['baseline'].shape
    appearance_checks = {
        'matching_rgb_shapes': reference_shape[-1] == 3
        and all(image.shape == reference_shape for image in preview_images.values()),
        'uint8_pixels': all(image.dtype == np.uint8 for image in preview_images.values()),
        'finite_pixels': all(np.isfinite(image).all() for image in preview_images.values()),
        'nonempty_frames': all(float(np.std(image)) > 0.0 for image in preview_images.values()),
        'sampled_observation_changed': any(
            not np.array_equal(preview_images['baseline'], preview_images[label])
            for label in ('seed 0', 'seed 1')
        ),
    }
    failed_appearance = [
        name for name, passed in appearance_checks.items() if not passed
    ]
    if failed_appearance:
        raise AssertionError(
            'Appearance evidence checks failed: ' + ', '.join(failed_appearance)
        )

    appearance_summary = {
        label: {
            'shape': list(image.shape),
            'mean_rgb': np.mean(image, axis=(0, 1)).round(2).tolist(),
            'pixel_std': round(float(np.std(image)), 2),
        }
        for label, image in preview_images.items()
    }
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, (label, image) in zip(axes, preview_images.items(), strict=True):
        axis.imshow(image)
        axis.set_title(label)
        axis.axis('off')
    figure.suptitle(f'Same task, Layer-A preview profile={PREVIEW_PROFILE!r}')
    figure.tight_layout()
    plt.show()
    print(json.dumps(appearance_summary, indent=2))
    print('Different pixels establish observation change, not realism or useful coverage.')
else:
    print('SKIP — no rendered appearance evidence in diagnostic mode')

## 4. Check Layer-B physics guardrails

Genesis combines the two surfaces of a contact pair using a maximum rule, so changing only one surface can be masked by the other. The current implementation applies one shared ratio to objects, Franka links, and the table. Mass uses a positive multiplicative ratio relative to pristine base mass, so resets do not compound.

The camera calculation belongs to the sensor domain even though the static world-camera pose is applied during reset. This lesson does not randomize the wrist-camera mounting transform.

In [ ]:
base_contact_friction = {'object': 0.4, 'finger': 0.8, 'table': 0.6}
candidate_ratios = np.asarray([0.75, 1.25])
object_only_effective = np.asarray(
    [
        max(base_contact_friction['object'] * ratio, base_contact_friction['finger'])
        for ratio in candidate_ratios
    ]
)
shared_effective = (
    max(base_contact_friction.values()) * candidate_ratios
)

base_masses_kg = np.asarray([0.06, 0.12, 0.18])
mass_ratios = np.asarray([0.8, 1.0, 1.2])
mass_shifts_kg = base_masses_kg * (mass_ratios - 1.0)
updated_masses_kg = base_masses_kg + mass_shifts_kg
replayed_masses_kg = base_masses_kg + mass_shifts_kg

guardrail_checks = {
    # Changing only object friction can be hidden by a larger finger friction.
    'object_only_can_be_masked': np.isclose(
        object_only_effective[0], object_only_effective[1]
    ),
    # Scaling all contact participants changes the effective pair friction.
    'shared_ratio_changes_pair_friction': not np.isclose(
        shared_effective[0], shared_effective[1]
    ),
    # Positive multiplicative ratios keep every randomized mass physical.
    'positive_multiplicative_mass': np.all(updated_masses_kg > 0.0),
    # Replaying a reset recomputes mass from the pristine base, without drift.
    'mass_references_pristine_base': np.allclose(
        replayed_masses_kg, base_masses_kg * mass_ratios
    ),
    # Both configured world-camera pose jitter amplitudes are enabled.
    'world_camera_only': collection_config['cam_pos_jitter'] > 0.0
    and collection_config['cam_lookat_jitter'] > 0.0,
}
failed_guardrails = [name for name, passed in guardrail_checks.items() if not passed]
if failed_guardrails:
    raise AssertionError('Physics guardrail checks failed: ' + ', '.join(failed_guardrails))

print('Contact-pair max(), object-only ratios:', object_only_effective.tolist())
print('Contact-pair max(), shared ratios:', shared_effective.round(3).tolist())
for base, ratio, shift, updated in zip(
    base_masses_kg, mass_ratios, mass_shifts_kg, updated_masses_kg, strict=True
):
    print(
        f'base={base:.3f} kg × ratio={ratio:.2f} → '
        f'shift={shift:+.3f} kg, updated={updated:.3f} kg'
    )
print('Camera note: this lab jitters the static world camera, not the wrist mount.')

## 5. Record a four-episode combined smoke

The subprocess composes Layer A and Layer B: four accepted banana-to-bowl episodes, at most ten attempts, 5 FPS, 640×360 world/wrist H.264 video, two successes per appearance build, and bounded runtime friction, mass, and world-camera jitter. This creates two appearance domains with two episodes each, making the split contrast observable.

This is an integration smoke, not a coverage study. Failed attempts remain outside the demonstration samples but must appear in the provenance sidecar.

In [ ]:
record_command = [
    sys.executable,
    '-m',
    'robo_genesis.record_dataset',
    '--episodes',
    str(TARGET_EPISODES),
    '--max-attempts',
    str(MAX_ATTEMPTS),
    '--seed',
    str(BASE_SEED),
    '--fps',
    str(DATASET_FPS),
    '--img-width',
    str(IMG_WH[0]),
    '--img-height',
    str(IMG_WH[1]),
    '--vcodec',
    'h264',
    '--pick',
    '011_banana',
    '--repo-id',
    DR_REPO_ID,
    '--output-dir',
    str(dataset_root),
    '--dr-appearance',
    '--dr-object-color',
    '--dr-table-jitter',
    '0.15',
    '--dr-fov-jitter',
    '2.0',
    '--dr-rebuild-every',
    str(DR_REBUILD_EVERY),
    '--dr-runtime',
    '--dr-friction',
    '0.7',
    '1.3',
    '--dr-mass',
    '0.8',
    '1.2',
    '--dr-cam-pos',
    '0.01',
    '--dr-cam-lookat',
    '0.02',
]
if use_cpu_backend:
    record_command.append('--cpu')
if overwrite_enabled and dataset_root.exists():
    record_command.append('--overwrite')

record_checks = {
    'command_uses_installed_module': record_command[:3]
    == [sys.executable, '-m', 'robo_genesis.record_dataset'],
    'bounded_attempts': record_command[
        record_command.index('--max-attempts') + 1
    ] == '10',
    'small_h264_dataset': all(
        value in record_command for value in ('5', '640', '360', 'h264')
    ),
    'four_episodes_two_per_appearance_domain': TARGET_EPISODES == 4
    and collection_config['dr_rebuild_every'] == 2,
    'combined_layer_a_and_b': '--dr-appearance' in record_command
    and '--dr-runtime' in record_command,
}

print('Recorder command:', ' '.join(record_command))
if render_enabled:
    subprocess.run(record_command, check=True, env=os.environ.copy())
    provenance_path = dataset_root / DOMAIN_PROVENANCE_RELATIVE_PATH
    record_checks = {
        **record_checks,
        'dataset_info_written': (dataset_root / 'meta' / 'info.json').is_file(),
        'provenance_written': provenance_path.is_file(),
    }
    failed_record = [name for name, passed in record_checks.items() if not passed]
    if failed_record:
        raise AssertionError('Recording checks failed: ' + ', '.join(failed_record))
else:
    record_checks['diagnostic_mode_requested'] = True
    provenance_path = dataset_root / DOMAIN_PROVENANCE_RELATIVE_PATH
    print('SKIP — recorder subprocess was not started; no dataset was created')

## 6. Reopen baseline and DR data

Read both datasets through LeRobot metadata and PyAV, then display identified world/wrist samples. The comparison checks identity, schema, codec, and readable observations; it does not train or compare a policy.

In [ ]:
distribution_checks = {'diagnostic_mode_requested': not render_enabled}
dataset_reports = {}
dataset_samples = {}

if render_enabled:
    import matplotlib.pyplot as plt
    import torch
    from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

    def tensor_to_numpy(value):
        if isinstance(value, torch.Tensor):
            return value.detach().cpu().numpy()
        return np.asarray(value)

    def open_dataset_report(repo_id, root):
        metadata = LeRobotDatasetMetadata(repo_id, root=root)
        episodes = [metadata.episodes[index] for index in range(len(metadata.episodes))]
        first = episodes[0]
        middle_index = int(first['dataset_from_index']) + int(first['length']) // 2
        dataset = LeRobotDataset(repo_id, root=root, video_backend='pyav')
        sample = dataset[middle_index]
        images = {
            key: tensor_to_numpy(sample[key]).transpose(1, 2, 0)
            for key in metadata.video_keys
        }
        video_report = {}
        for key in metadata.video_keys:
            specification = metadata.features[key]
            details = specification.get('info') or specification.get('video_info') or {}
            video_report[key] = {
                'shape': list(specification['shape']),
                'codec': details.get('video.codec', 'unreported'),
                'fps': details.get('video.fps', 'unreported'),
            }
        report = {
            'root': str(metadata.root.resolve()),
            'codebase_version': metadata.info.codebase_version,
            'fps': metadata.fps,
            'episodes': metadata.total_episodes,
            'frames': metadata.total_frames,
            'camera_keys': sorted(metadata.video_keys),
            'video': video_report,
            'sample_episode': int(sample['episode_index']),
            'sample_frame': int(sample['frame_index']),
            'sample_mean_rgb': {
                key: np.mean(image, axis=(0, 1)).round(3).tolist()
                for key, image in images.items()
            },
        }
        return metadata, dataset, sample, images, report

    baseline_result = open_dataset_report(BASELINE_REPO_ID, baseline_root)
    dr_result = open_dataset_report(DR_REPO_ID, dataset_root)
    baseline_metadata, baseline_dataset, _, baseline_images, dataset_reports['baseline'] = (
        baseline_result
    )
    dr_metadata, dr_dataset, _, dr_images, dataset_reports['domain_randomized'] = dr_result
    dataset_samples = {'baseline': baseline_images, 'domain randomized': dr_images}

    expected_camera_keys = {
        'observation.images.world',
        'observation.images.wrist',
    }
    distribution_checks = {
        'baseline_has_two_episodes': baseline_metadata.total_episodes >= 2,
        'dr_has_requested_episodes': dr_metadata.total_episodes == TARGET_EPISODES,
        'both_camera_contracts': set(baseline_metadata.video_keys)
        == set(dr_metadata.video_keys)
        == expected_camera_keys,
        'matching_fps': baseline_metadata.fps == dr_metadata.fps == DATASET_FPS,
        'finite_decoded_images': all(
            np.isfinite(image).all()
            for sample_images in dataset_samples.values()
            for image in sample_images.values()
        ),
        'nonempty_decoded_images': all(
            float(np.std(image)) > 0.0
            for sample_images in dataset_samples.values()
            for image in sample_images.values()
        ),
    }
    failed_distribution = [
        name for name, passed in distribution_checks.items() if not passed
    ]
    if failed_distribution:
        raise AssertionError(
            'Dataset audit checks failed: ' + ', '.join(failed_distribution)
        )

    camera_order = sorted(expected_camera_keys)
    figure, axes = plt.subplots(2, 2, figsize=(10, 8))
    for row, (label, images) in enumerate(dataset_samples.items()):
        for column, key in enumerate(camera_order):
            axes[row, column].imshow(np.clip(images[key], 0.0, 1.0))
            axes[row, column].set_title(f'{label}: {key.rsplit(".", 1)[-1]}')
            axes[row, column].axis('off')
    figure.suptitle('Persisted baseline and domain-randomized samples')
    figure.tight_layout()
    plt.show()
    print(json.dumps(dataset_reports, indent=2))
    print('These frames establish readable observations, not policy robustness.')
else:
    print('SKIP — no LeRobot metadata, samples, or videos were opened')

## 7. Join episodes to provenance and inspect split risk

The sidecar preserves every attempted domain, including attempts rejected by the success gate. Join committed episodes to their appearance/runtime identities, check actual values against requested bounds, and plan IDs without copying the dataset.

Four committed episodes are grouped as domains `0, 0, 1, 1`. The in-distribution plan places one episode from each domain on each side; the held-out-domain plan keeps each complete appearance group on one side. This demonstrates split mechanics—not coverage, model selection, or a benchmark.

In [ ]:
provenance_checks = {'diagnostic_mode_requested': not render_enabled}
split_reports = {}
attempt_rows = []

if render_enabled:
    import matplotlib.pyplot as plt

    provenance = load_domain_provenance(dataset_root)
    episode_domain_records = committed_episode_records(provenance)
    in_distribution_split = plan_domain_split(
        episode_domain_records,
        eval_fraction=0.5,
        mode='in_distribution',
    )
    held_out_split = plan_domain_split(
        episode_domain_records,
        eval_fraction=0.5,
        mode='held_out_domain',
    )
    split_reports = {
        'in_distribution': in_distribution_split,
        'held_out_domain': held_out_split,
    }
    attempt_rows = provenance['attempts']
    requested_runtime = provenance['requested']['runtime']
    friction_low, friction_high = requested_runtime['friction_ratio_range']
    mass_low, mass_high = requested_runtime['mass_ratio_range']

    actual_values_in_range = True
    camera_values_in_range = True
    for attempt in attempt_rows:
        actual = attempt['actual']
        friction = actual['friction_ratio']
        masses = tuple(actual['mass_ratios'].values())
        camera = actual['world_camera_pose']
        actual_values_in_range &= friction_low <= friction <= friction_high
        actual_values_in_range &= bool(masses) and all(
            mass_low <= ratio <= mass_high for ratio in masses
        )
        camera_values_in_range &= camera is not None
        if camera is not None:
            from robo_genesis.scene_config import WORLD_CAM_LOOKAT, WORLD_CAM_POS

            camera_values_in_range &= np.all(
                np.abs(np.asarray(camera['pos']) - np.asarray(WORLD_CAM_POS))
                <= requested_runtime['cam_pos_jitter'] + 1e-12
            )
            camera_values_in_range &= np.all(
                np.abs(np.asarray(camera['lookat']) - np.asarray(WORLD_CAM_LOOKAT))
                <= requested_runtime['cam_lookat_jitter'] + 1e-12
            )

    committed_ids = [
        record['committed_episode_index'] for record in episode_domain_records
    ]
    provenance_checks = {
        'complete_collection': provenance['summary']['complete'],
        'requested_episode_count': len(episode_domain_records) == TARGET_EPISODES,
        'dataset_episode_alignment': committed_ids
        == list(range(dr_metadata.total_episodes)),
        'attempt_count_matches': provenance['summary']['attempts'] == len(attempt_rows),
        'runtime_seed_schedule': all(
            row['runtime_seed'] == BASE_SEED + row['attempt_index']
            for row in attempt_rows
        ),
        'actual_physics_in_range': actual_values_in_range,
        'actual_world_camera_in_range': camera_values_in_range,
        'in_distribution_domains_shared': set(
            in_distribution_split['train_domain_ids']
        ) == set(in_distribution_split['eval_domain_ids'])
        and bool(in_distribution_split['overlapping_domain_ids']),
        'held_out_domains_disjoint': not held_out_split['overlapping_domain_ids'],
        'split_modes_are_distinct': in_distribution_split['train_episode_ids']
        != held_out_split['train_episode_ids'],
        'split_covers_committed_episodes': held_out_split['all_episodes_covered'],
    }
    failed_provenance = [
        name for name, passed in provenance_checks.items() if not passed
    ]
    if failed_provenance:
        raise AssertionError(
            'Provenance checks failed: ' + ', '.join(failed_provenance)
        )

    attempt_indices = np.asarray(
        [row['attempt_index'] for row in attempt_rows], dtype=int
    )
    friction_ratios = np.asarray(
        [row['actual']['friction_ratio'] for row in attempt_rows], dtype=float
    )
    domain_indices = np.asarray(
        [row['appearance_domain_index'] for row in attempt_rows], dtype=int
    )
    success_mask = np.asarray([row['success'] for row in attempt_rows], dtype=bool)
    figure, axis = plt.subplots(figsize=(10, 4.5))
    axis.scatter(
        attempt_indices[success_mask],
        friction_ratios[success_mask],
        label='committed success',
        color='#0891b2',
        marker='o',
    )
    axis.scatter(
        attempt_indices[~success_mask],
        friction_ratios[~success_mask],
        label='failed attempt',
        color='#f97316',
        marker='x',
    )
    axis.set_xlabel('attempt index')
    axis.set_ylabel('shared friction ratio')
    axis.set_title('Every attempted domain remains visible in provenance')
    domain_axis = axis.twinx()
    domain_axis.step(
        attempt_indices,
        domain_indices,
        where='mid',
        color='#8b5cf6',
        alpha=0.55,
        label='appearance domain',
    )
    domain_axis.set_ylabel('appearance domain index')
    handles, labels = axis.get_legend_handles_labels()
    domain_handles, domain_labels = domain_axis.get_legend_handles_labels()
    axis.legend(handles + domain_handles, labels + domain_labels, loc='best')
    figure.tight_layout()
    plt.show()

    print('Attempt trace:')
    print(json.dumps(attempt_rows, indent=2))
    print('Split comparison (episode IDs grouped by appearance domain):')
    for label, report in split_reports.items():
        print(
            f"  {label}: train={report['train_episode_ids']} / "
            f"domains={report['train_domain_ids']}; "
            f"eval={report['eval_episode_ids']} / "
            f"domains={report['eval_domain_ids']}; "
            f"shared domains={report['overlapping_domain_ids']}"
        )
    print('Full split-plan details:')
    print(json.dumps(split_reports, indent=2))
    print('The split helper plans IDs only; it does not copy data or configure training.')
else:
    print('SKIP — no provenance was read and no dataset split was claimed')

## 8. Final evidence boundary

A passing full path means the current DR knobs, recorder, sidecar, reader, and split validator connect. A diagnostic pass means only pure logic and command construction were checked.

In [ ]:
if render_enabled:
    final_checks = {
        'manifest_contract': lesson.status.value == 'planned'
        and lesson.duration_minutes == 90,
        'backend_contract': resolved_backend in {'cpu', 'amdgpu'},
        'randomization_contract': all(contract_checks.values()),
        'appearance_preview': all(preview_checks.values()),
        'appearance_evidence': all(appearance_checks.values()),
        'physics_guardrails': all(guardrail_checks.values()),
        'recording_and_sidecar': all(record_checks.values()),
        'dataset_readback': all(distribution_checks.values()),
        'provenance_and_split': all(provenance_checks.values()),
    }
    failed_final = [name for name, passed in final_checks.items() if not passed]
    if failed_final:
        raise AssertionError('L11 final checks failed: ' + ', '.join(failed_final))
    print(
        f'L11 CHECK: PASSED — backend={resolved_backend}; '
        f'{len(episode_domain_records)} committed episodes; '
        f'{len(attempt_rows)} attempts audited'
    )
    print('Training: NOT RUN')
    print('Closed-loop policy evaluation: NOT RUN')
    print('Sim-to-real improvement: NOT MEASURED')
else:
    diagnostic_checks = {
        'manifest_contract': lesson.status.value == 'planned'
        and lesson.duration_minutes == 90,
        'randomization_contract': all(contract_checks.values()),
        'preview_command_constructed': preview_checks['command_uses_installed_module'],
        'physics_guardrails': all(guardrail_checks.values()),
        'record_command_constructed': record_checks['command_uses_installed_module'],
        'preview_not_run': preview_checks['diagnostic_mode_requested'],
        'recorder_not_run': record_checks['diagnostic_mode_requested'],
        'reader_not_run': distribution_checks['diagnostic_mode_requested'],
        'provenance_not_read': provenance_checks['diagnostic_mode_requested'],
    }
    failed_diagnostic = [
        name for name, passed in diagnostic_checks.items() if not passed
    ]
    if failed_diagnostic:
        raise AssertionError(
            'L11 diagnostic checks failed: ' + ', '.join(failed_diagnostic)
        )
    print('L11 DIAGNOSTIC CHECK: PASSED')
    print('Core domain-randomization experiment: NOT COMPLETED')
    print('No Genesis scene, preview, dataset, provenance, training, or evaluation was run.')

## One-variable exercise and checkpoints

Return to the `l11-appearance-preview` code cell in Section 2. Change only `PREVIEW_PROFILE` from `combined` to `table_color`, `object_color`, or `fov`, then rerun Sections 2 and 3. The profile fixes every unselected knob at zero, keeps seeds 0 and 1, writes to an isolated preview subdirectory, and displays the DR-off baseline beside the selected variation. Static world-camera pose is excluded because it is a reset-time Layer-B parameter and requires an episode recording rather than this build-time preview.

Use the outputs to answer three closing questions:

1. After changing one `PREVIEW_PROFILE`, what changed, what stayed fixed, and what can the resulting frames actually establish?
2. For appearance domains `0, 0, 1, 1`, what different evaluation question does each split plan answer?
3. Why should provenance retain failed attempts that were not written to the dataset?